In [ ]:
%%sql -r dataframe_2
USE ROLE PROJECT_MEMBER;
USE WAREHOUSE FINAL_PROJECT_WH;
USE DATABASE FINAL_PROJ_DB;
USE SCHEMA RAW_DATA;
--INTERGATION SETUP IN SETUP.IPYNB
DESC INTEGRATION GCS_INTEGRATION;


In [ ]:
%%sql -r dataframe_3
--DESC INTEGRATION GCS_INTEGRATION;

CREATE FILE FORMAT WEATHER_CSV_RAW
    TYPE = 'CSV'
    COMPRESSION = 'GZIP'         -- Tells Snowflake to decompress natively
    FIELD_DELIMITER = ','        -- Standard comma separation
    RECORD_DELIMITER = '\n'
    NULL_IF = ('NULL', 'null')
    SKIP_HEADER = 1
    FIELD_OPTIONALLY_ENCLOSED_BY = '"';

CREATE STAGE final_proj_stage
URL = 'gcs://nvms_final_project_bucket/data/'
STORAGE_INTEGRATION = gcs_integration
FILE_FORMAT = WEATHER_CSV_RAW;

LIST @final_proj_stage;


In [ ]:
%%sql -r dataframe_1
CREATE TABLE RAW_STORM_EVENTS (
BEGIN_YEARMONTH VARCHAR,
BEGIN_DAY VARCHAR,
BEGIN_TIME VARCHAR,
END_YEARMONTH VARCHAR,
END_DAY VARCHAR,
END_TIME VARCHAR,
EPISODE_ID VARCHAR,
EVENT_ID VARCHAR,
STATE VARCHAR,
STATE_FIPS VARCHAR, -- STATE_FIPS_CODE 
YEAR VARCHAR,
MONTH_NAME VARCHAR,
EVENT_TYPE VARCHAR,
CZ_TYPE VARCHAR,
CZ_FIPS VARCHAR,
CZ_NAME VARCHAR,
WFO VARCHAR,
BEGIN_DATETIME VARCHAR,
CZ_TIMEZONE VARCHAR,
END_DATE_TIME VARCHAR,
INJURIES_DIRECT VARCHAR,
INJURIES_INDIRECT VARCHAR,
DEATHS_DIRECT VARCHAR,
DEATHS_INDIRECT VARCHAR,
DAMAGE_PROPERTY VARCHAR,
DAMAGE_CROPS VARCHAR,
SOURCE VARCHAR,
MAGNITUDE VARCHAR,
MAGNITUDE_TYPE VARCHAR,
FLOOD_CAUSE VARCHAR,
CATEGORY VARCHAR,
TOR_F_SCALE VARCHAR,
TOR_LENGTH VARCHAR,
TOR_WIDTH VARCHAR,
TOR_OTHER_WFO VARCHAR,
TOR_OTHER_CZ_STATE VARCHAR,
TOR_OTHER_CZ_FIPS VARCHAR,
TOR_OTHER_CZ_NAME VARCHAR,
BEGIN_RANGE VARCHAR,
BEGIN_AZIMUTH VARCHAR,
BEGIN_LOCATION VARCHAR,
END_RANGE VARCHAR,
END_AZIMUTH VARCHAR,
END_LOCATION VARCHAR,
BEGIN_LAT VARCHAR, 
BEGIN_LON VARCHAR,
END_LAT VARCHAR,
END_LON VARCHAR,
EPISODE_NARRATIVE VARCHAR,
EVENT_NARRATIVE VARCHAR,
DATA_SOURCE VARCHAR,
_FILE_NAME VARCHAR,
_LOADED_AT TIMESTAMP
);

DESCRIBE TABLE RAW_STORM_EVENTS;


In [ ]:
%%sql -r dataframe_8


SELECT CURRENT_DATABASE(), CURRENT_SCHEMA();
USE SCHEMA RAW_DATA;
CREATE TABLE RAW_STORM_EVENTS_TEST LIKE RAW_STORM_EVENTS;
--SHOW OBJECTS LIKE 'RAW_STORM_EVENTS_TEST';

--DROP TABLE RAW_DATA.RAW_STORM_EVENTS_TEST;


In [ ]:
%%sql -r dataframe_4
/*COPY INTO RAW_STORM_EVENTS
FROM( 
SELECT
$1::VARCHAR,
$2::VARCHAR,
$3::VARCHAR,
$4::VARCHAR,
$5::VARCHAR,
$6::VARCHAR,
$7::VARCHAR,
$8::VARCHAR,
$9::VARCHAR,
$10::VARCHAR,
$11::VARCHAR,
$12::VARCHAR,
$13::VARCHAR,
$14::VARCHAR,
$15::VARCHAR,
$16::VARCHAR,
$17::VARCHAR,
$18::VARCHAR,
$19::VARCHAR,
$20::VARCHAR,
$21::VARCHAR,
$22::VARCHAR,
$23::VARCHAR,
$24::VARCHAR,
$25::VARCHAR,
$26::VARCHAR,
$27::VARCHAR,
$28::VARCHAR,
$29::VARCHAR,
$30::VARCHAR,
$31::VARCHAR,
$32::VARCHAR,
$33::VARCHAR,
$34::VARCHAR,
$35::VARCHAR,
$36::VARCHAR,
$37::VARCHAR,
$38::VARCHAR,
$39::VARCHAR,
$40::VARCHAR,
$41::VARCHAR,
$42::VARCHAR,
$43::VARCHAR,
$44::VARCHAR,
$45::VARCHAR,
$46::VARCHAR,
$47::VARCHAR,
$48::VARCHAR,
$49::VARCHAR,
$50::VARCHAR,
$51::VARCHAR,
SPLIT_PART(METADATA$FILENAME, '/', 3), -- Captures file name
CURRENT_TIMESTAMP(),  
FROM @FINAL_PROJ_STAGE/
)
FILE_FORMAT = WEATHER_CSV_RAW;

*/


In [ ]:
%%sql -r dataframe_9
CREATE OR REPLACE NOTIFICATION INTEGRATION PUB_SUB_INTEGRATION
  TYPE = QUEUE
  NOTIFICATION_PROVIDER = GCP_PUBSUB
  ENABLED = TRUE
  GCP_PUBSUB_SUBSCRIPTION_NAME = 'projects/project-ccadea78-681f-4b30-b55/subscriptions/noaa-file-arrivals-topic-sub';

 DESC NOTIFICATION INTEGRATION PUB_SUB_INTEGRATION;

In [ ]:
%%sql -r dataframe_12
CREATE OR REPLACE PIPE STORM_EVENTS_SNOWPIPE
AUTO_INGEST = TRUE
INTEGRATION = 'PUB_SUB_INTEGRATION'
AS
COPY INTO RAW_STORM_EVENTS
FROM( 
SELECT
$1::VARCHAR,
$2::VARCHAR,
$3::VARCHAR,
$4::VARCHAR,
$5::VARCHAR,
$6::VARCHAR,
$7::VARCHAR,
$8::VARCHAR,
$9::VARCHAR,
$10::VARCHAR,
$11::VARCHAR,
$12::VARCHAR,
$13::VARCHAR,
$14::VARCHAR,
$15::VARCHAR,
$16::VARCHAR,
$17::VARCHAR,
$18::VARCHAR,
$19::VARCHAR,
$20::VARCHAR,
$21::VARCHAR,
$22::VARCHAR,
$23::VARCHAR,
$24::VARCHAR,
$25::VARCHAR,
$26::VARCHAR,
$27::VARCHAR,
$28::VARCHAR,
$29::VARCHAR,
$30::VARCHAR,
$31::VARCHAR,
$32::VARCHAR,
$33::VARCHAR,
$34::VARCHAR,
$35::VARCHAR,
$36::VARCHAR,
$37::VARCHAR,
$38::VARCHAR,
$39::VARCHAR,
$40::VARCHAR,
$41::VARCHAR,
$42::VARCHAR,
$43::VARCHAR,
$44::VARCHAR,
$45::VARCHAR,
$46::VARCHAR,
$47::VARCHAR,
$48::VARCHAR,
$49::VARCHAR,
$50::VARCHAR,
$51::VARCHAR,
SPLIT_PART(METADATA$FILENAME, '/', 3),
CURRENT_TIMESTAMP() 
FROM @FINAL_PROJ_STAGE/
)
FILE_FORMAT = WEATHER_CSV_RAW
ON_ERROR = SKIP_FILE;


In [ ]:
%%sql -r dataframe_11
ALTER PIPE STORM_EVENTS_SNOWPIPE REFRESH;

In [ ]:
%%sql -r dataframe_10
SHOW PIPES;
SELECT SYSTEM$PIPE_STATUS(
    'FINAL_PROJ_DB.RAW_DATA.STORM_EVENTS_SNOWPIPE'
);


In [ ]:
%%sql -r dataframe_13
SELECT * FROM TABLE(INFORMATION_SCHEMA.PIPE_USAGE_HISTORY(PIPE_NAME=>'STORM_EVENTS_SNOWPIPE'));


In [ ]:
%%sql -r dataframe_7
--for testing
SELECT COUNT('*')
FROM RAW_STORM_EVENTS_TEST;

In [ ]:
%%sql -r dataframe_5
--exploration ON RAW DATA
/*
select episode_id, event_id from 
raw_storm_events
where cz_fips is null or state_fips is null;
*/
--SELECT * FROM RAW_STORM_EVENTS WHERE BEGIN_DATETIME > END_DATE_TIME;
/*THE ADDITIONAL DATA COLUMNS ARE : 

--EXPLORING EVENT TYPE DATA
SELECT * FROM 
( SELECT *, ROW_NUMBER() OVER 
    (PARTITION BY EVENT_TYPE ORDER BY BEGIN_YEARMONTH DESC)as rn
    FROM RAW_STORM_EVENTS ) WHERE rn < 3;

--CHECK WHERE BEGIN_LAT IS NULL and not null FOR HOW MANY EPISODES
SELECT COUNT(EPISODE_ID),EVENT_TYPE FROM RAW_STORM_EVENTS WHERE BEGIN_LAT IS NOT NULL OR TRIM(BEGIN_LAT) <> '' 
GROUP BY EVENT_TYPE;
*/


In [ ]:
%%sql -r dataframe_6
USE SCHEMA SILVER;
--EXPLORING SILVER TABLE
/*
 SELECT EPISODE_ID,EVENT_ID,EVENT_TYPE,CZ_TYPE,CZ_NAME,CZ_FIPS,STATE_FIPS,STATE_ABBR,STATE, BEGIN_DATETIME_LOCAL_TS, BEGIN_DATETIME_UTC_TS,END_DATETIME_LOCAL_TS, END_DATETIME_UTC_TS, DAMAGE_CROPS, DAMAGE_PROPERTY, INJURIES_DIRECT,INJURIES_INDIRECT,DEATHS_DIRECT,DEATHS_INDIRECT,LOADED_AT,_FILE_NAME
 FROM INTERMEDIATE_STAGING_TABLE
 WHERE STATE IS NULL OR TRIM(STATE) = '';

 */